- build geometry with build_module_boxes
- convert your mask into fiducial boxes with fiducialize_boxes_from_mask
- fast batched MC generator:
- - continuous sampling (uniform in fiducial volumes)
- - voxelized sampling (snap to grid to emulate lattice spikes)
- - applies min_dist cut while still returning exactly N events
- computes angles with the same dz-ordering convention as data
- loads your HDF5 data, applies the same selection, and overlays plots

# Cell 0 — imports

In [1]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
    TQDM_OK = True
except Exception:
    TQDM_OK = False
import os
import re
from pathlib import Path
import math


### Cell 0.1 — figure saving helpers

In [2]:

# Cell 0.1 — figure saving helpers
OUTPUT_DIR = Path("double_blip_figures")
SAVE_FIGURES = True
FIG_DPI = 180

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figures will be saved to: {OUTPUT_DIR.resolve()}")

_FIG_SAVE_COUNTER = {"n": 0}

def _slugify(text):
    text = str(text)
    text = re.sub(r"\$+", "", text)
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text).strip("._")
    return text or "figure"

def save_current_figure(name, fig=None, dpi=FIG_DPI, close=False):
    if not SAVE_FIGURES:
        return None
    _FIG_SAVE_COUNTER["n"] += 1
    stem = f"{_FIG_SAVE_COUNTER['n']:03d}_{_slugify(name)}"
    out = OUTPUT_DIR / f"{stem}.png"
    if fig is None:
        fig = plt.gcf()
    fig.savefig(out, dpi=dpi, bbox_inches="tight")
    print(f"Saved: {out}")
    if close:
        plt.close(fig)
    return out


Figures will be saved to: /home/bruno/combinatoryBg/double_blip_figures


# Cell 1 — geometry helpers

In [3]:
def build_module_boxes(Lx=60.0, Ly=120.0, Lz=60.0, gapx=6.0, gapz=5.0):
    """
    Build 2x2 modules centered at (+/-xoff, +/-zoff) with true face-to-face gaps gapx, gapz.
    Returns list of boxes dicts with xmin/xmax/ymin/ymax/zmin/zmax and module id.
    """
    hx, hy, hz = Lx/2, Ly/2, Lz/2
    xoff = (Lx + gapx)/2
    zoff = (Lz + gapz)/2

    boxes = []
    mid = 0
    for sx in (+1, -1):      # right, left
        for sz in (+1, -1):  # +z, -z
            boxes.append({
                "module": mid,
                "xmin": sx*xoff - hx,
                "xmax": sx*xoff + hx,
                "ymin": -hy,
                "ymax": +hy,
                "zmin": sz*zoff - hz,
                "zmax": sz*zoff + hz,
            })
            mid += 1
    return boxes


def fiducialize_boxes_from_mask(module_boxes, y_half=51.85, z_inner_abs=12.68, z_outer_abs=54.32):
    """
    Intersect each module box with:
      y in [-y_half, +y_half]
      z in [ z_inner_abs,  z_outer_abs] for +z modules
      z in [-z_outer_abs, -z_inner_abs] for -z modules
    """
    fid_boxes = []
    for b in module_boxes:
        fymin = max(b["ymin"], -y_half)
        fymax = min(b["ymax"], +y_half)
        if fymin >= fymax:
            continue

        zc = 0.5 * (b["zmin"] + b["zmax"])
        if zc >= 0:
            z_keep_min, z_keep_max = z_inner_abs, z_outer_abs
        else:
            z_keep_min, z_keep_max = -z_outer_abs, -z_inner_abs

        fzmin = max(b["zmin"], z_keep_min)
        fzmax = min(b["zmax"], z_keep_max)
        if fzmin >= fzmax:
            continue

        fid_boxes.append({
            "module": b["module"],
            "xmin": b["xmin"],
            "xmax": b["xmax"],
            "ymin": fymin,
            "ymax": fymax,
            "zmin": fzmin,
            "zmax": fzmax,
        })
    return fid_boxes

# Cell 2 — fast sampling from fiducial boxes (continuous + voxelized)

In [4]:
def _boxes_to_arrays(boxes):
    xmin = np.array([b["xmin"] for b in boxes], dtype=np.float32)
    xmax = np.array([b["xmax"] for b in boxes], dtype=np.float32)
    ymin = np.array([b["ymin"] for b in boxes], dtype=np.float32)
    ymax = np.array([b["ymax"] for b in boxes], dtype=np.float32)
    zmin = np.array([b["zmin"] for b in boxes], dtype=np.float32)
    zmax = np.array([b["zmax"] for b in boxes], dtype=np.float32)
    mod  = np.array([b["module"] for b in boxes], dtype=np.int32)
    vol  = (xmax-xmin)*(ymax-ymin)*(zmax-zmin)
    p    = vol / vol.sum()
    return xmin,xmax,ymin,ymax,zmin,zmax,mod,p


def sample_hits_in_boxes(rng, n, boxes_arrays, voxel_size=None):
    """
    Sample n points uniformly from a union of axis-aligned boxes, weighted by volume.
    If voxel_size is not None (float), snap x/y/z to a cubic voxel grid of that size.
    """
    xmin,xmax,ymin,ymax,zmin,zmax,mod,p = boxes_arrays
    k = rng.choice(len(p), size=n, replace=True, p=p)

    x = rng.uniform(xmin[k], xmax[k]).astype(np.float32)
    y = rng.uniform(ymin[k], ymax[k]).astype(np.float32)
    z = rng.uniform(zmin[k], zmax[k]).astype(np.float32)
    m = mod[k]

    if voxel_size is not None:
        vs = float(voxel_size)
        x = (np.round(x / vs) * vs).astype(np.float32)
        y = (np.round(y / vs) * vs).astype(np.float32)
        z = (np.round(z / vs) * vs).astype(np.float32)

        # Clamp back into box bounds (rounding can push you slightly outside)
        x = np.clip(x, xmin[k], xmax[k]).astype(np.float32)
        y = np.clip(y, ymin[k], ymax[k]).astype(np.float32)
        z = np.clip(z, zmin[k], zmax[k]).astype(np.float32)

    return x, y, z, m

# Cell 3 — angles computed exactly like your data (dz forced positive)

In [5]:
def angles_like_data(ax, ay, az, bx, by, bz):
    """
    Match data convention:
    - order the two points by z (so dz >= 0)
    - theta_z  = arccos(dz/r)
    - theta_zx = atan2(dx, dz)
    - theta_zy = atan2(dy, dz)
    """
    dz = bz - az
    swap = dz < 0
    if np.any(swap):
        ax2, ay2, az2 = ax.copy(), ay.copy(), az.copy()
        bx2, by2, bz2 = bx.copy(), by.copy(), bz.copy()
        ax2[swap], bx2[swap] = bx2[swap], ax2[swap]
        ay2[swap], by2[swap] = by2[swap], ay2[swap]
        az2[swap], bz2[swap] = bz2[swap], az2[swap]
        ax, ay, az, bx, by, bz = ax2, ay2, az2, bx2, by2, bz2

    dx = bx - ax
    dy = by - ay
    dz = bz - az
    r  = np.sqrt(dx*dx + dy*dy + dz*dz)

    theta_z  = np.arccos(dz / r)
    theta_zx = np.arctan2(dx, dz)
    theta_zy = np.arctan2(dy, dz)
    return theta_z.astype(np.float32), theta_zx.astype(np.float32), theta_zy.astype(np.float32)

# Cell 4 — batched generator with min_dist cut (returns exactly N)

In [6]:
def generate_doublets_batched(
    boxes_arrays,
    N=5_000_000,
    seed=123,
    batch_size=500_000,
    min_dist=10.0,
    voxel_size=None,
    store_max=2000,
    show_progress=True,
    oversample=1.30,
    return_r=False,
):
    rng = np.random.default_rng(seed)

    theta_z  = np.empty(N, dtype=np.float32)
    theta_zx = np.empty(N, dtype=np.float32)
    theta_zy = np.empty(N, dtype=np.float32)
    r_out = np.empty(N, dtype=np.float32) if return_r else None

    # sample indices to store (global)
    if store_max > 0:
        store_idx = rng.choice(N, size=min(store_max, N), replace=False)
        store_idx.sort()
    else:
        store_idx = np.array([], dtype=np.int64)
    stored_tracks = []
    store_ptr = 0

    # tqdm that updates every batch (and also every refill loop)
    pbar = None
    if show_progress and TQDM_OK:
        pbar = tqdm(total=N, desc="Generating doublets", unit="evt")

    accepted_total = 0
    tried_total = 0

    n_batches = (N + batch_size - 1) // batch_size
    for bi in range(n_batches):
        start = bi * batch_size
        end = min(N, (bi + 1) * batch_size)
        nb = end - start

        # preallocate for this batch
        ax = np.empty(nb, dtype=np.float32)
        ay = np.empty(nb, dtype=np.float32)
        az = np.empty(nb, dtype=np.float32)
        am = np.empty(nb, dtype=np.int32)

        bx = np.empty(nb, dtype=np.float32)
        by = np.empty(nb, dtype=np.float32)
        bz = np.empty(nb, dtype=np.float32)
        bm = np.empty(nb, dtype=np.int32)

        fill = 0
        while fill < nb:
            need = nb - fill
            m = int(np.ceil(need * oversample))

            ax0, ay0, az0, am0 = sample_hits_in_boxes(rng, m, boxes_arrays, voxel_size=voxel_size)
            bx0, by0, bz0, bm0 = sample_hits_in_boxes(rng, m, boxes_arrays, voxel_size=voxel_size)

            dx = bx0 - ax0
            dy = by0 - ay0
            dz = bz0 - az0
            dist = np.sqrt(dx*dx + dy*dy + dz*dz)

            tried_total += m
            ok = dist >= float(min_dist)
            n_ok = int(ok.sum())
            if n_ok == 0:
                # still update tqdm occasionally so it doesn't look "stuck"
                if pbar is not None:
                    pbar.set_postfix(acc=f"{accepted_total/max(tried_total,1):.3f}")
                    pbar.update(0)
                continue

            take = min(n_ok, need)
            idx_ok = np.flatnonzero(ok)[:take]

            ax[fill:fill+take] = ax0[idx_ok]
            ay[fill:fill+take] = ay0[idx_ok]
            az[fill:fill+take] = az0[idx_ok]
            am[fill:fill+take] = am0[idx_ok]

            bx[fill:fill+take] = bx0[idx_ok]
            by[fill:fill+take] = by0[idx_ok]
            bz[fill:fill+take] = bz0[idx_ok]
            bm[fill:fill+take] = bm0[idx_ok]

            fill += take
            accepted_total += take

            if pbar is not None:
                pbar.update(take)
                pbar.set_postfix(acc=f"{accepted_total/max(tried_total,1):.3f}")

        # angles (same as data convention)
        thz, thzx, thzy = angles_like_data(ax, ay, az, bx, by, bz)
        theta_z[start:end]  = thz
        theta_zx[start:end] = thzx
        theta_zy[start:end] = thzy

        if return_r:
            dx = bx - ax; dy = by - ay; dz = bz - az
            r_out[start:end] = np.sqrt(dx*dx + dy*dy + dz*dz).astype(np.float32)

        # store tracks for overlays (unchanged logic)
        while store_ptr < len(store_idx) and store_idx[store_ptr] < end:
            gi = store_idx[store_ptr]
            li = gi - start
            stored_tracks.append({
                "a": {"x": float(ax[li]), "y": float(ay[li]), "z": float(az[li]), "module": int(am[li])},
                "b": {"x": float(bx[li]), "y": float(by[li]), "z": float(bz[li]), "module": int(bm[li])},
                "theta_z": float(thz[li]),
                "theta_zx": float(thzx[li]),
                "theta_zy": float(thzy[li]),
            })
            store_ptr += 1

    if pbar is not None:
        pbar.close()

    if return_r:
        return theta_z, theta_zx, theta_zy, r_out, stored_tracks
    return theta_z, theta_zx, theta_zy, stored_tracks

# Cell 5 — data loader (matches selection; optional centroid)

In [7]:
def load_data_angles(
    h5_path,
    min_dist=10.0,
    use_centroid=False,
):
    theta_z_real = []
    theta_zx_real = []
    theta_zy_real = []

    with h5py.File(h5_path, "r") as f:
        for key in f["events"]:
            g = f["events"][key]
            labels = g["labels"][:]
            x = g["x"][:]
            y = g["y"][:]
            z = g["z"][:]

            geom_mask = (
                (y >= -51.85) & (y <= 51.85) &
                (((z >= 12.68) & (z <= 54.32)) | ((z >= -54.32) & (z <= -12.68)))
            )

            idx = np.where(geom_mask & (labels >= 0))[0]
            if idx.size < 2:
                continue

            labs = labels[idx]
            uniq = np.unique(labs)
            if uniq.size != 2:
                continue

            c0 = idx[labs == uniq[0]]
            c1 = idx[labs == uniq[1]]

            if use_centroid:
                x2 = np.array([x[c0].mean(), x[c1].mean()])
                y2 = np.array([y[c0].mean(), y[c1].mean()])
                z2 = np.array([z[c0].mean(), z[c1].mean()])
            else:
                # match your current behavior
                idx0 = c0[0]
                idx1 = c1[0]
                x2 = np.array([x[idx0], x[idx1]])
                y2 = np.array([y[idx0], y[idx1]])
                z2 = np.array([z[idx0], z[idx1]])

            dx = x2[1] - x2[0]
            dy = y2[1] - y2[0]
            dz = z2[1] - z2[0]
            dist = np.sqrt(dx*dx + dy*dy + dz*dz)
            if dist < float(min_dist):
                continue

            # same convention as data function: order by z
            order = np.argsort(z2)
            x2, y2, z2 = x2[order], y2[order], z2[order]
            dx, dy, dz = x2[1]-x2[0], y2[1]-y2[0], z2[1]-z2[0]
            r = np.sqrt(dx*dx + dy*dy + dz*dz)

            theta_z_real.append(np.arccos(dz / r))
            theta_zx_real.append(np.arctan2(dx, dz))
            theta_zy_real.append(np.arctan2(dy, dz))

    return (np.array(theta_z_real),
            np.array(theta_zx_real),
            np.array(theta_zy_real))

# Cell 6 — plotting overlay helpers

In [8]:
def hist_with_err(data, nbins=60, range=None):
    counts, edges = np.histogram(data, bins=nbins, range=range)
    centers = 0.5*(edges[1:] + edges[:-1])
    err = np.sqrt(counts)
    return counts, centers, edges, err


def chi2_ndf_pvalue_from_hists(data_counts, model_counts, n_fit_params=1, min_variance=1.0):
    """
    Simple binned chi2 using Poisson data variance.

    Parameters
    ----------
    data_counts : array-like
        Observed bin counts.
    model_counts : array-like
        Expected / model bin counts, already normalized as desired.
    n_fit_params : int
        Number of fitted parameters to subtract from NDF.
        Use 1 when the model is normalized to the data.
    min_variance : float
        Floor for the variance to avoid divide-by-zero in empty bins.

    Returns
    -------
    chi2, ndf, pval, mask
    """
    data_counts = np.asarray(data_counts, dtype=float)
    model_counts = np.asarray(model_counts, dtype=float)

    # Keep bins where at least one side has content
    mask = (data_counts > 0) | (model_counts > 0)
    if not np.any(mask):
        return np.nan, 0, np.nan, mask

    d = data_counts[mask]
    m = model_counts[mask]
    var = np.maximum(d, float(min_variance))

    chi2 = float(np.sum((d - m)**2 / var))
    ndf = int(mask.sum() - n_fit_params)
    if ndf <= 0:
        return chi2, ndf, np.nan, mask

    pval = float(scipy.stats.chi2.sf(chi2, ndf))
    return chi2, ndf, pval, mask


def _stats_label(chi2, ndf, pval):
    if not np.isfinite(chi2) or ndf <= 0 or not np.isfinite(pval):
        return "chi2/ndf = n/a, p = n/a"
    return fr"$\chi^2/\mathrm{{ndf}} = {chi2:.1f}/{ndf} = {chi2/ndf:.2f}$,  $p = {pval:.3g}$"


def overlay_hist(mc, data_new, data_old, nbins=60, title="", xlabel="angle (rad)", scale_to="data_new"):
    # Common binning from the union of all finite entries for stable overlay
    mc = np.asarray(mc)
    data_new = np.asarray(data_new)
    data_old = np.asarray(data_old)

    mc = mc[np.isfinite(mc)]
    data_new = data_new[np.isfinite(data_new)]
    data_old = data_old[np.isfinite(data_old)]

    if len(mc) == 0 or len(data_new) == 0 or len(data_old) == 0:
        print(f"Skipping {title}: one of the inputs is empty.")
        return

    finite_all = np.concatenate([mc, data_new, data_old])
    data_range = (float(np.min(finite_all)), float(np.max(finite_all)))

    counts_d, centers, edges, err_d = hist_with_err(data_new, nbins=nbins, range=data_range)
    counts_d_old, centers_old, edges_old, err_d_old = hist_with_err(data_old, nbins=nbins, range=data_range)
    counts_m, _ = np.histogram(mc, bins=edges)

    if scale_to == "data_new":
        s = counts_d.sum() / max(counts_m.sum(), 1)
    elif scale_to == "data_old":
        s = counts_d_old.sum() / max(counts_m.sum(), 1)
    elif scale_to == "mc":
        s = 1.0
    else:
        s = float(scale_to)

    # For visual comparison, scale old clustering to the same total as new clustering
    size_old = counts_d_old.sum()
    size_new = counts_d.sum()
    scale_factor_old = size_new / max(size_old, 1)

    counts_d_old_vis = counts_d_old.astype(float) * scale_factor_old
    err_d_old_vis = err_d_old.astype(float) * scale_factor_old
    model_scaled = counts_m.astype(float) * s

    # Goodness-of-fit for each data set against the same MC model
    chi2_new, ndf_new, p_new, _ = chi2_ndf_pvalue_from_hists(counts_d, model_scaled, n_fit_params=1)
    chi2_old, ndf_old, p_old, _ = chi2_ndf_pvalue_from_hists(counts_d_old, model_scaled, n_fit_params=1)

    plt.figure(figsize=(10, 5))
    plt.hist(mc, bins=edges, histtype="step", label=f"MC (scaled x{s:.3g})",
             weights=np.ones_like(mc, dtype=float) * s)
    plt.errorbar(
        centers, counts_d, yerr=err_d, fmt="o", markersize=4,
        label=f"data_new | {_stats_label(chi2_new, ndf_new, p_new)}"
    )
    plt.errorbar(
        centers_old, counts_d_old_vis, yerr=err_d_old_vis, fmt="o", markersize=4,
        label=f"data_old (renorm.) | {_stats_label(chi2_old, ndf_old, p_old)}"
    )
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Counts")
    plt.legend(fontsize=9)
    plt.tight_layout()
    save_current_figure(title or "overlay_hist")
    plt.show()

    print(f"{title}")
    print(f"  new clustering: {_stats_label(chi2_new, ndf_new, p_new)}")
    print(f"  old clustering: {_stats_label(chi2_old, ndf_old, p_old)}")

# Cell 7 — run everything (continuous + voxelized)

In [9]:
# Geometry
MODULE_BOXES = build_module_boxes(Lx=60, Ly=120, Lz=60, gapx=6, gapz=5)
FID_BOXES    = fiducialize_boxes_from_mask(MODULE_BOXES, y_half=51.85, z_inner_abs=12.68, z_outer_abs=54.32)
boxes_arrays = _boxes_to_arrays(FID_BOXES)

print("Modules:", len(MODULE_BOXES), " Fiducial boxes:", len(FID_BOXES))
for b in FID_BOXES:
    print(b)

min_dist = 10.0

# MC continuous
theta_z_mc, theta_zx_mc, theta_zy_mc, stored_tracks = generate_doublets_batched(
    boxes_arrays,
    N=20_000_000,
    seed=1,
    batch_size=400_000,
    min_dist=min_dist,
    voxel_size=None,     # continuous
    store_max=200000,
    show_progress=True,
)

# MC voxelized (optional; reduce N if slow)
theta_z_vox, theta_zx_vox, theta_zy_vox, _ = generate_doublets_batched(
    boxes_arrays,
    N=10_000_000,
    seed=2,
    batch_size=250_000,
    min_dist=min_dist,
    voxel_size=0.4,      # 1 cm voxel grid (tune)
    store_max=0,
    show_progress=True,
)

Modules: 4  Fiducial boxes: 4
{'module': 0, 'xmin': 3.0, 'xmax': 63.0, 'ymin': -51.85, 'ymax': 51.85, 'zmin': 12.68, 'zmax': 54.32}
{'module': 1, 'xmin': 3.0, 'xmax': 63.0, 'ymin': -51.85, 'ymax': 51.85, 'zmin': -54.32, 'zmax': -12.68}
{'module': 2, 'xmin': -63.0, 'xmax': -3.0, 'ymin': -51.85, 'ymax': 51.85, 'zmin': 12.68, 'zmax': 54.32}
{'module': 3, 'xmin': -63.0, 'xmax': -3.0, 'ymin': -51.85, 'ymax': 51.85, 'zmin': -54.32, 'zmax': -12.68}


Generating doublets:   0%|          | 0/20000000 [00:00<?, ?evt/s]

Generating doublets:   0%|          | 0/10000000 [00:00<?, ?evt/s]

# Cell 7.1 - load data (slow)

In [10]:
# Data
theta_z_real, theta_zx_real, theta_zy_real = load_data_angles(
    "normal_clusters.h5",
    min_dist=min_dist,
    use_centroid=True,   # try True later!
)
print("Data doublets:", len(theta_z_real), " (new clusters)")

# Data - old clusters
theta_z_old, theta_zx_old, theta_zy_old = load_data_angles(
    "HC/test_hot.h5",
    min_dist=min_dist,
    use_centroid=True,   # try True later!
)
print("Data doublets:", len(theta_z_old), " (old clusters)")

KeyError: "Unable to synchronously open object (object 'events' doesn't exist)"

### Cell 7.2 — multiplicity / Poisson / single-blip resampling helpers

This section adds three things on top of the original double-blip study:

1. **Cluster multiplicity per event** for each dataset, with Poisson overlays.
2. **A singles-driven combinatorial MC**, where synthetic doublets are built by sampling from the observed **single-cluster** population instead of from a uniform spatial prior.
3. **A simple Ar39 rate estimate** for the fiducial volume, so the observed multiplicity can be compared with a physically motivated baseline.

A few implementation notes:
- The fiducial selection follows the same `y/z` acceptance already used above.
- "Single-blip distribution" means clusters taken from events with exactly **one accepted cluster** in the fiducial volume.
- Charge / energy is loaded in a best-effort way by looking for common charge-like dataset names in each event group. If no compatible quantity is found, the rate/geometry studies still run and the energy plots are skipped.
- The Ar39 comparison needs an **event window duration**. Leave `EVENT_WINDOW_S = None` to skip that overlay, or set it to the readout window used to build these event files.


In [ ]:
import scipy.stats

# Cell 7.2 — multiplicity / Poisson / single-blip resampling helpers

AR39_ACTIVITY_BQ_PER_KG = 0.97      # atmospheric argon, Bq / kg
LAR_DENSITY_G_PER_CM3 = 1.3954      # liquid argon near 87 K
EVENT_WINDOW_S = 20e-6               # e.g. 2.2e-3 if you want an Ar39 overlay
AR39_RECO_EFFICIENCY = 1.0          # optional effective acceptance / reconstruction factor
SINGLES_MC_TARGET = 200_000         # accepted synthetic doublets to build from singles
SINGLE_ENERGY_KEYS = [
    "energy", "E", "adc", "adcs", "q", "Q", "charge", "charges", "dq", "de", "nhit", "nhits"
]


def fiducial_geom_mask_from_arrays(x, y, z):
    return (
        (y >= -51.85) & (y <= 51.85) &
        (((z >= 12.68) & (z <= 54.32)) | ((z >= -54.32) & (z <= -12.68)))
    )


def infer_cluster_energy(event_group, cluster_indices, candidate_keys=SINGLE_ENERGY_KEYS):
    """
    Best-effort cluster energy / charge estimator.

    Returns the sum over the first charge-like dataset found in the event group.
    If nothing suitable is present, returns np.nan.
    """
    n_points = len(event_group["x"])

    for key in candidate_keys:
        if key not in event_group:
            continue

        arr = event_group[key][:]
        arr = np.asarray(arr)

        # Per-point quantity: sum over hits in the cluster
        if arr.ndim == 1 and len(arr) == n_points:
            try:
                return float(np.nansum(arr[cluster_indices]))
            except Exception:
                pass

        # Already one value per event / cluster-like array — not safe to assume mapping here
        # so we skip it rather than risk a wrong interpretation.

    return np.nan


def _event_time_from_group(group):
    """
    Try to recover an event time / integration window from the group.
    This is only a helper; returning None is perfectly fine.
    """
    attr_candidates = [
        "event_window_s", "window_s", "readout_window_s", "dt_s",
        "duration_s", "livetime_s", "trigger_window_s"
    ]
    for k in attr_candidates:
        if k in group.attrs:
            try:
                return float(group.attrs[k])
            except Exception:
                pass
    return None


def load_cluster_event_summary(h5_path, use_centroid=True):
    """
    Parse all events and summarize accepted fiducial clusters.

    Returns a dict with:
      - multiplicities: number of accepted clusters per event
      - singles: pool of clusters from events with multiplicity == 1
      - doublets: accepted events with multiplicity == 2
      - triplets: accepted events with multiplicity == 3
      - all_clusters: all accepted fiducial clusters regardless of event multiplicity
      - event_windows_s: optional event windows recovered from attrs when present
    """
    multiplicities = []
    singles = []
    doublets = []
    triplets = []
    all_clusters = []
    event_windows_s = []

    with h5py.File(h5_path, "r") as f:
        events = f["events"]
        keys = list(events.keys())

        iterable = tqdm(keys, desc=f"Summarizing {Path(h5_path).name}") if TQDM_OK else keys
        for key in iterable:
            g = events[key]
            labels = np.asarray(g["labels"][:])
            x = np.asarray(g["x"][:], dtype=float)
            y = np.asarray(g["y"][:], dtype=float)
            z = np.asarray(g["z"][:], dtype=float)

            geom_mask = fiducial_geom_mask_from_arrays(x, y, z)
            idx = np.where(geom_mask & (labels >= 0))[0]
            event_time = _event_time_from_group(g)
            if event_time is not None:
                event_windows_s.append(event_time)

            if idx.size == 0:
                multiplicities.append(0)
                continue

            labs = labels[idx]
            uniq = np.unique(labs)
            clusters = []
            for lab in uniq:
                cidx = idx[labs == lab]
                if cidx.size == 0:
                    continue

                if use_centroid:
                    pos = np.array([x[cidx].mean(), y[cidx].mean(), z[cidx].mean()], dtype=float)
                else:
                    pos = np.array([x[cidx[0]], y[cidx[0]], z[cidx[0]]], dtype=float)

                energy = infer_cluster_energy(g, cidx)
                cluster = {
                    "event_key": key,
                    "label": int(lab),
                    "npts": int(cidx.size),
                    "x": float(pos[0]),
                    "y": float(pos[1]),
                    "z": float(pos[2]),
                    "energy": float(energy) if np.isfinite(energy) else np.nan,
                }
                clusters.append(cluster)
                all_clusters.append(cluster)

            multiplicities.append(len(clusters))

            if len(clusters) == 1:
                singles.append(clusters[0])
            elif len(clusters) == 2:
                doublets.append(clusters)
            elif len(clusters) == 3:
                triplets.append(clusters)

    return {
        "multiplicities": np.asarray(multiplicities, dtype=int),
        "singles": singles,
        "doublets": doublets,
        "triplets": triplets,
        "all_clusters": all_clusters,
        "event_windows_s": np.asarray(event_windows_s, dtype=float),
    }


def poisson_pmf(k, lam):
    """
    Numerically stable Poisson PMF using log-space:
        P(k; lam) = exp(-lam + k*log(lam) - lgamma(k+1))
    """
    k = np.asarray(k, dtype=int)
    out = np.zeros_like(k, dtype=float)

    if lam is None or lam < 0:
        return out

    if lam == 0:
        out[k == 0] = 1.0
        return out

    for i, kk in enumerate(k):
        out[i] = math.exp(-lam + kk * math.log(lam) - math.lgamma(kk + 1))

    return out

def fiducial_volume_cm3(boxes):
    vol = 0.0
    for b in boxes:
        vol += (b["xmax"] - b["xmin"]) * (b["ymax"] - b["ymin"]) * (b["zmax"] - b["zmin"])
    return float(vol)


def ar39_lambda_for_window(boxes, event_window_s, activity_bq_per_kg=AR39_ACTIVITY_BQ_PER_KG,
                           density_g_per_cm3=LAR_DENSITY_G_PER_CM3, reco_efficiency=AR39_RECO_EFFICIENCY):
    vol_cm3 = fiducial_volume_cm3(boxes)
    mass_kg = vol_cm3 * density_g_per_cm3 / 1000.0
    rate_hz = activity_bq_per_kg * mass_kg
    lam = rate_hz * float(event_window_s) * float(reco_efficiency)
    return lam, mass_kg, vol_cm3, rate_hz


def plot_multiplicity_poisson(multiplicities, title, lam_ref=None, lam_ref_label=None, kmax_show=None):
    multiplicities = np.asarray(multiplicities)

    if len(multiplicities) == 0:
        print(f"No events found for {title}")
        return

    if kmax_show is None:
        # safer default than max(multiplicities), which can be dominated by outliers
        kmax_show = max(8, int(np.percentile(multiplicities, 99.5)))
        kmax_show = min(kmax_show, int(np.max(multiplicities)))

    ks = np.arange(0, kmax_show + 1)
    counts = np.array([(multiplicities == k).sum() for k in ks], dtype=float)
    errs = np.sqrt(counts)
    n_evt = len(multiplicities)
    lam_hat = float(np.mean(multiplicities))

    plt.figure(figsize=(8, 5))
    plt.errorbar(ks, counts, yerr=errs, fmt="o", capsize=3, label=f"Observed ({n_evt} events)")

    model_obs = n_evt * poisson_pmf(ks, lam_hat)
    plt.plot(ks, model_obs, linewidth=2, label=fr"Poisson with $\lambda=\langle n \rangle={lam_hat:.3f}$")

    if lam_ref is not None:
        model_ref = n_evt * poisson_pmf(ks, lam_ref)
        lbl = lam_ref_label or fr"Reference Poisson ($\lambda={lam_ref:.3f}$)"
        plt.plot(ks, model_ref, "--", linewidth=2, label=lbl)

    plt.xlabel("accepted clusters per event")
    plt.ylabel("number of events")
    plt.title(title)
    plt.yscale("log")
    plt.ylim(bottom=0.1) 
    plt.legend()
    plt.tight_layout()
    save_current_figure(title)
    plt.show()

    frac = lambda n: float(np.mean(multiplicities == n))
    tail2 = float(np.mean(multiplicities >= 2))
    tail3 = float(np.mean(multiplicities >= 3))
    print(f"{title}")
    print(f"  mean multiplicity = {lam_hat:.5f}")
    print(f"  P(n=0,1,2,3) obs  = {frac(0):.5f}, {frac(1):.5f}, {frac(2):.5f}, {frac(3):.5f}")
    print(f"  P(n>=2), P(n>=3) = {tail2:.5f}, {tail3:.5f}")
    if lam_ref is not None:
        pmf_ref = poisson_pmf(np.arange(4), lam_ref)
        print(f"  Ref lambda        = {lam_ref:.5f}")
        print(f"  Ref P(n=0,1,2,3)  = {pmf_ref[0]:.5f}, {pmf_ref[1]:.5f}, {pmf_ref[2]:.5f}, {pmf_ref[3]:.5f}")

def clusters_to_arrays(clusters):
    if len(clusters) == 0:
        return {
            "x": np.array([]), "y": np.array([]), "z": np.array([]),
            "energy": np.array([]), "npts": np.array([])
        }
    return {
        "x": np.array([c["x"] for c in clusters], dtype=float),
        "y": np.array([c["y"] for c in clusters], dtype=float),
        "z": np.array([c["z"] for c in clusters], dtype=float),
        "energy": np.array([c["energy"] for c in clusters], dtype=float),
        "npts": np.array([c["npts"] for c in clusters], dtype=float),
    }


def build_doublet_arrays_from_cluster_pairs(cluster_pairs, min_dist=10.0, sort_by_z=True):
    dx, dy, dz = [], [], []
    theta_z, theta_zx, theta_zy = [], [], []
    e0, e1, esum, eabsdiff = [], [], [], []
    npts0, npts1 = [], []

    for c0, c1 in cluster_pairs:
        p0 = np.array([c0["x"], c0["y"], c0["z"]], dtype=float)
        p1 = np.array([c1["x"], c1["y"], c1["z"]], dtype=float)
        if sort_by_z and (p1[2] < p0[2]):
            p0, p1 = p1, p0
            c0, c1 = c1, c0

        d = p1 - p0
        dist = np.linalg.norm(d)
        if dist < float(min_dist):
            continue

        dx.append(d[0]); dy.append(d[1]); dz.append(d[2])
        theta_z.append(np.arccos(np.clip(d[2] / dist, -1.0, 1.0)))
        theta_zx.append(np.arctan2(d[0], d[2]))
        theta_zy.append(np.arctan2(d[1], d[2]))

        ee0 = c0.get("energy", np.nan)
        ee1 = c1.get("energy", np.nan)
        e0.append(ee0); e1.append(ee1)
        esum.append(ee0 + ee1 if np.isfinite(ee0) and np.isfinite(ee1) else np.nan)
        eabsdiff.append(abs(ee1 - ee0) if np.isfinite(ee0) and np.isfinite(ee1) else np.nan)
        npts0.append(c0.get("npts", np.nan)); npts1.append(c1.get("npts", np.nan))

    return {
        "dx": np.asarray(dx), "dy": np.asarray(dy), "dz": np.asarray(dz),
        "theta_z": np.asarray(theta_z), "theta_zx": np.asarray(theta_zx), "theta_zy": np.asarray(theta_zy),
        "e0": np.asarray(e0, dtype=float), "e1": np.asarray(e1, dtype=float),
        "esum": np.asarray(esum, dtype=float), "eabsdiff": np.asarray(eabsdiff, dtype=float),
        "npts0": np.asarray(npts0, dtype=float), "npts1": np.asarray(npts1, dtype=float),
    }


def monte_carlo_doublets_from_singles(single_clusters, n_accept=100_000, min_dist=10.0, sort_by_z=True,
                                      seed=1234, allow_same_source_event=True, max_trials_factor=50):
    """
    Build synthetic doublets by independently drawing from the observed single-cluster pool.
    """
    rng = np.random.default_rng(seed)
    singles = list(single_clusters)
    if len(singles) < 2:
        raise ValueError("Need at least two single clusters to build a singles-driven MC.")

    accepted_pairs = []
    trials = 0
    max_trials = int(max_trials_factor * n_accept)

    while len(accepted_pairs) < n_accept and trials < max_trials:
        trials += 1
        i0 = rng.integers(0, len(singles))
        i1 = rng.integers(0, len(singles) - 1)
        if i1 >= i0:
            i1 += 1
        c0 = singles[i0]
        c1 = singles[i1]

        if (not allow_same_source_event) and (c0["event_key"] == c1["event_key"]):
            continue

        p0 = np.array([c0["x"], c0["y"], c0["z"]], dtype=float)
        p1 = np.array([c1["x"], c1["y"], c1["z"]], dtype=float)
        if np.linalg.norm(p1 - p0) < float(min_dist):
            continue

        accepted_pairs.append((c0, c1))

    if len(accepted_pairs) < n_accept:
        print(f"Warning: requested {n_accept} accepted singles-MC doublets, got {len(accepted_pairs)}")

    return build_doublet_arrays_from_cluster_pairs(accepted_pairs, min_dist=min_dist, sort_by_z=sort_by_z)


def overlay_hist_simple(
    data_a, data_b, label_a, label_b, title, xlabel,
    bins=60, density=True, range=None, annotate_stats=True
):
    a = np.asarray(data_a)
    b = np.asarray(data_b)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0:
        print(f"Skipping {title}: one of the inputs is empty.")
        return

    # Common binning from the union of both samples
    if range is None:
        allv = np.concatenate([a, b])
        range = (float(np.min(allv)), float(np.max(allv)))

    counts_a, edges = np.histogram(a, bins=bins, range=range)
    counts_b, _ = np.histogram(b, bins=edges)
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths = np.diff(edges)

    # Scale model / MC to the observed total count for chi2 and visual comparison
    scale_b = counts_a.sum() / max(counts_b.sum(), 1)
    model_b = counts_b.astype(float) * scale_b

    chi2, ndf, pval, _ = chi2_ndf_pvalue_from_hists(counts_a, model_b, n_fit_params=1)

    # Plot as normalized counts or raw counts, but always with Poisson data bars
    if density:
        norm_a = max(len(a), 1) * widths
        norm_b = max(len(b), 1) * widths
        y_a = counts_a / norm_a
        yerr_a = np.sqrt(counts_a) / norm_a
        y_b = counts_b / norm_b
        ylabel = "normalized counts"
    else:
        y_a = counts_a.astype(float)
        yerr_a = np.sqrt(counts_a)
        y_b = model_b
        ylabel = "counts"

    plt.figure(figsize=(7, 5))
    plt.errorbar(
        centers, y_a, yerr=yerr_a, fmt="o", markersize=4, capsize=2,
        label=f"{label_a} | {_stats_label(chi2, ndf, pval)}"
    )

    if density:
        plt.hist(
            b, bins=edges, density=True, histtype="step", linewidth=2,
            linestyle="--", label=label_b
        )
    else:
        plt.step(edges[:-1], y_b, where="post", linestyle="--", linewidth=2, label=label_b)

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(fontsize=9)
    plt.tight_layout()
    save_current_figure(title)
    plt.show()

    print(f"{title}")
    print(f"  {_stats_label(chi2, ndf, pval)}")


# Cell 7.3 — build multiplicity summaries for both datasets

In [ ]:
# Cell 7.3 — build multiplicity summaries for both datasets
summary_new = load_cluster_event_summary("normal_clusters.h5", use_centroid=True)
summary_old = load_cluster_event_summary("HC/test_hot.h5", use_centroid=True)

print("New clustering:")
print("  total events   =", len(summary_new["multiplicities"]))
print("  singles pool   =", len(summary_new["singles"]))
print("  doublets       =", len(summary_new["doublets"]))
print("  triplets       =", len(summary_new["triplets"]))

print("Old clustering:")
print("  total events   =", len(summary_old["multiplicities"]))
print("  singles pool   =", len(summary_old["singles"]))
print("  doublets       =", len(summary_old["doublets"]))
print("  triplets       =", len(summary_old["triplets"]))


### Cell 7.4 — Poisson multiplicity plots and Ar39 comparison

The solid curve uses the **observed** mean multiplicity for each dataset. If `EVENT_WINDOW_S` is set, the dashed curve shows the Poisson expectation from Ar39 in the fiducial volume using the simple activity × mass × window estimate.


In [ ]:
# Cell 7.4 — Poisson multiplicity plots and Ar39 comparison
lam_ar39 = None
if EVENT_WINDOW_S is not None:
    lam_ar39, fid_mass_kg, fid_vol_cm3, ar39_rate_hz = ar39_lambda_for_window(
        FID_BOXES,
        EVENT_WINDOW_S,
        activity_bq_per_kg=AR39_ACTIVITY_BQ_PER_KG,
        density_g_per_cm3=LAR_DENSITY_G_PER_CM3,
        reco_efficiency=AR39_RECO_EFFICIENCY,
    )
    print(f"Fiducial volume [cm^3] = {fid_vol_cm3:.2f}")
    print(f"Fiducial mass   [kg]   = {fid_mass_kg:.4f}")
    print(f"Ar39 decay rate [Hz]   = {ar39_rate_hz:.4f}")
    print(f"Ar39 lambda / event    = {lam_ar39:.6f}   (for EVENT_WINDOW_S = {EVENT_WINDOW_S})")
else:
    print("EVENT_WINDOW_S is None, so the explicit Ar39 overlay is skipped.")

plot_multiplicity_poisson(
    summary_new["multiplicities"],
    title="Multiplicity per event — new clustering",
    lam_ref=lam_ar39,
    lam_ref_label=(fr"Ar39 expectation ($\lambda={lam_ar39:.4g}$)" if lam_ar39 is not None else None),
)

plot_multiplicity_poisson(
    summary_old["multiplicities"],
    title="Multiplicity per event — old DBSCAN clustering",
    lam_ref=lam_ar39,
    lam_ref_label=(fr"Ar39 expectation ($\lambda={lam_ar39:.4g}$)" if lam_ar39 is not None else None),
)


### Cell 7.5 — build the observed doublet samples from cluster summaries

These arrays are reconstructed directly from the parsed cluster summaries, so they are also convenient for the singles-driven resampling study.


In [ ]:
# Cell 7.5 — build the observed doublet samples from cluster summaries
obs_doublets_new = build_doublet_arrays_from_cluster_pairs(summary_new["doublets"], min_dist=min_dist, sort_by_z=True)
obs_doublets_old = build_doublet_arrays_from_cluster_pairs(summary_old["doublets"], min_dist=min_dist, sort_by_z=True)

print("Observed new-clustering doublets after min_dist:", len(obs_doublets_new["dx"]))
print("Observed old-clustering doublets after min_dist:", len(obs_doublets_old["dx"]))


### Cell 7.6 — singles-driven combinatorial MC

Instead of drawing two blips uniformly in the fiducial volume, this MC draws two independent clusters from the **observed singles population**. That lets you test whether the doublet sample can be explained by independent combinations of the measured single-blip distribution.


In [ ]:
# Cell 7.6 — singles-driven combinatorial MC
singles_mc_new = monte_carlo_doublets_from_singles(
    summary_new["singles"],
    n_accept=SINGLES_MC_TARGET,
    min_dist=min_dist,
    sort_by_z=True,
    seed=10,
    allow_same_source_event=False,
)

singles_mc_old = monte_carlo_doublets_from_singles(
    summary_old["singles"],
    n_accept=SINGLES_MC_TARGET,
    min_dist=min_dist,
    sort_by_z=True,
    seed=20,
    allow_same_source_event=False,
)

print("Singles-driven MC (new singles pool):", len(singles_mc_new["dx"]))
print("Singles-driven MC (old singles pool):", len(singles_mc_old["dx"]))


### Cell 7.7 — geometry / angular comparison: observed doublets vs singles-driven MC

In [ ]:
# Cell 7.7 — geometry / angular comparison: observed doublets vs singles-driven MC
def plot_delta_occupancy_2d(a, b, xlabel, ylabel, title, bins=80, symmetric=True, equal_aspect=True):
    plt.figure(figsize=(7, 6))

    if len(a) == 0 or len(b) == 0:
        print(f"No entries for: {title}")
        return

    if symmetric:
        lim = max(np.max(np.abs(a)), np.max(np.abs(b)))
        ranges = [[-lim, lim], [-lim, lim]]
    else:
        ranges = None

    plt.hist2d(a, b, bins=bins, range=ranges, cmap="viridis")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.colorbar(label="counts")
    if equal_aspect:
        plt.gca().set_aspect("equal", adjustable="box")
    plt.tight_layout()
    save_current_figure(title)
    plt.show()


def plot_all_delta_occupancies(dx, dy, dz, prefix="Sample", bins=80, symmetric=True):
    plot_delta_occupancy_2d(
        dx, dz,
        xlabel=r"$\Delta x$ [cm]",
        ylabel=r"$\Delta z$ [cm]",
        title=f"{prefix}: $\Delta x$ vs $\Delta z$",
        bins=bins,
        symmetric=symmetric,
    )

    plot_delta_occupancy_2d(
        dz, dy,
        xlabel=r"$\Delta z$ [cm]",
        ylabel=r"$\Delta y$ [cm]",
        title=f"{prefix}: $\Delta z$ vs $\Delta y$",
        bins=bins,
        symmetric=symmetric,
    )

def projected_separations(dx, dy, dz):
    rho_xz = np.sqrt(dx*dx + dz*dz)
    rho_zy = np.sqrt(dz*dz + dy*dy)
    return rho_xz, rho_zy


def overlay_projected_separations(rho_xz_data, rho_zy_data, rho_xz_mc, rho_zy_mc, bins=50):
    fig, ax = plt.subplots(figsize=(8, 5))

    allv = np.concatenate([rho_xz_data, rho_zy_data, rho_xz_mc, rho_zy_mc])
    edges = np.linspace(float(np.min(allv)), float(np.max(allv)), bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths = np.diff(edges)

    def _density_with_err(data):
        counts, _ = np.histogram(data, bins=edges)
        dens = counts / (max(len(data), 1) * widths)
        err = np.sqrt(counts) / (max(len(data), 1) * widths)
        return counts, dens, err

    c_xz_d, d_xz_d, e_xz_d = _density_with_err(rho_xz_data)
    c_zy_d, d_zy_d, e_zy_d = _density_with_err(rho_zy_data)
    c_xz_m, _, _ = _density_with_err(rho_xz_mc)
    c_zy_m, _, _ = _density_with_err(rho_zy_mc)

    scale_xz = c_xz_d.sum() / max(c_xz_m.sum(), 1)
    scale_zy = c_zy_d.sum() / max(c_zy_m.sum(), 1)

    chi2_xz, ndf_xz, p_xz, _ = chi2_ndf_pvalue_from_hists(c_xz_d, c_xz_m * scale_xz, n_fit_params=1)
    chi2_zy, ndf_zy, p_zy, _ = chi2_ndf_pvalue_from_hists(c_zy_d, c_zy_m * scale_zy, n_fit_params=1)

    ax.errorbar(centers, d_xz_d, yerr=e_xz_d, fmt="o", markersize=4, capsize=2,
                label=fr"Data xz | {_stats_label(chi2_xz, ndf_xz, p_xz)}")
    ax.errorbar(centers, d_zy_d, yerr=e_zy_d, fmt="s", markersize=4, capsize=2,
                label=fr"Data zy | {_stats_label(chi2_zy, ndf_zy, p_zy)}")

    ax.hist(rho_xz_mc, bins=edges, density=True, histtype="step", linewidth=2, linestyle="--",
            label=r"MC xz")
    ax.hist(rho_zy_mc, bins=edges, density=True, histtype="step", linewidth=2, linestyle="--",
            label=r"MC zy")

    ax.set_xlabel("projected separation [cm]")
    ax.set_ylabel("normalized counts")
    ax.set_title("Projected doublet separation: data vs MC")
    ax.legend(fontsize=8)
    plt.tight_layout()
    save_current_figure("Projected_doublet_separation_data_vs_MC")
    plt.show()

    print("Projected doublet separation: data vs MC")
    print(f"  xz: {_stats_label(chi2_xz, ndf_xz, p_xz)}")
    print(f"  zy: {_stats_label(chi2_zy, ndf_zy, p_zy)}")

def two_panel_projected_comparison(rho_xz_data, rho_zy_data, rho_xz_mc, rho_zy_mc, bins=50):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    datasets = [
        (axes[0], rho_xz_data, rho_xz_mc, "xz projection", "projected separation in xz [cm]"),
        (axes[1], rho_zy_data, rho_zy_mc, "zy projection", "projected separation in zy [cm]"),
    ]

    for ax, data, mc, title, xlabel in datasets:
        allv = np.concatenate([data, mc])
        edges = np.linspace(float(np.min(allv)), float(np.max(allv)), bins + 1)
        centers = 0.5 * (edges[:-1] + edges[1:])
        widths = np.diff(edges)

        counts_d, _ = np.histogram(data, bins=edges)
        counts_m, _ = np.histogram(mc, bins=edges)

        dens_d = counts_d / (max(len(data), 1) * widths)
        err_d = np.sqrt(counts_d) / (max(len(data), 1) * widths)

        scale_m = counts_d.sum() / max(counts_m.sum(), 1)
        chi2, ndf, pval, _ = chi2_ndf_pvalue_from_hists(counts_d, counts_m * scale_m, n_fit_params=1)

        ax.errorbar(centers, dens_d, yerr=err_d, fmt="o", markersize=4, capsize=2,
                    label=f"Data | {_stats_label(chi2, ndf, pval)}")
        ax.hist(mc, bins=edges, density=True, histtype="step", linewidth=2, linestyle="--", label="MC")
        ax.set_xlabel(xlabel)
        ax.set_title(title)
        ax.legend(fontsize=8)

    axes[0].set_ylabel("normalized counts")
    plt.tight_layout()
    save_current_figure("Projected_separation_two_panel_comparison")
    plt.show()


plot_all_delta_occupancies(
    obs_doublets_new["dx"], obs_doublets_new["dy"], obs_doublets_new["dz"],
    prefix="Observed new doublets (from summary)", bins=70, symmetric=False,
)
plot_all_delta_occupancies(
    singles_mc_new["dx"], singles_mc_new["dy"], singles_mc_new["dz"],
    prefix="Singles-driven MC from new singles", bins=70, symmetric=False,
)

overlay_hist_simple(
    obs_doublets_new["theta_zx"], singles_mc_new["theta_zx"],
    "Observed new doublets", "Singles-driven MC",
    title="theta_zx — observed new doublets vs singles-driven MC",
    xlabel=r"$\theta_{zx}$ [rad]", bins=60,
)

overlay_hist_simple(
    obs_doublets_new["theta_zy"], singles_mc_new["theta_zy"],
    "Observed new doublets", "Singles-driven MC",
    title="theta_zy — observed new doublets vs singles-driven MC",
    xlabel=r"$\theta_{zy}$ [rad]", bins=60,
)

overlay_hist_simple(
    obs_doublets_new["theta_z"], singles_mc_new["theta_z"],
    "Observed new doublets", "Singles-driven MC",
    title="theta_z — observed new doublets vs singles-driven MC",
    xlabel=r"$\theta_{z}$ [rad]", bins=60,
)

rho_xz_obs_new, rho_zy_obs_new = projected_separations(obs_doublets_new["dx"], obs_doublets_new["dy"], obs_doublets_new["dz"])
rho_xz_smc_new, rho_zy_smc_new = projected_separations(singles_mc_new["dx"], singles_mc_new["dy"], singles_mc_new["dz"])

overlay_projected_separations(rho_xz_obs_new, rho_zy_obs_new, rho_xz_smc_new, rho_zy_smc_new, bins=50)
two_panel_projected_comparison(rho_xz_obs_new, rho_zy_obs_new, rho_xz_smc_new, rho_zy_smc_new, bins=50)


### Cell 7.8 — energy / cluster-size comparison: observed doublets vs singles-driven MC

If a charge-like per-hit quantity is available in the input HDF5, these plots compare the observed doublet energy structure against the singles-driven combinatorial expectation.


In [ ]:
# Cell 7.8 — energy / cluster-size comparison: observed doublets vs singles-driven MC
for label, obs, smc in [
    ("new clustering", obs_doublets_new, singles_mc_new),
    ("old DBSCAN clustering", obs_doublets_old, singles_mc_old),
]:
    print(f"\nEnergy/size comparison for {label}")

    overlay_hist_simple(
        obs["npts0"] + obs["npts1"], smc["npts0"] + smc["npts1"],
        f"Observed {label}", "Singles-driven MC",
        title=f"Total cluster size (npts) — {label}",
        xlabel="npts(cluster 1) + npts(cluster 2)", bins=50,
    )

    if np.isfinite(obs["esum"]).sum() > 0 and np.isfinite(smc["esum"]).sum() > 0:
        overlay_hist_simple(
            obs["esum"], smc["esum"],
            f"Observed {label}", "Singles-driven MC",
            title=f"Pair energy / charge sum — {label}",
            xlabel="E1 + E2 (arb.)", bins=60,
        )
        overlay_hist_simple(
            obs["eabsdiff"], smc["eabsdiff"],
            f"Observed {label}", "Singles-driven MC",
            title=f"|E2 - E1| — {label}",
            xlabel="|E2 - E1| (arb.)", bins=60,
        )
    else:
        print("  No usable per-cluster energy / charge field found; skipping energy plots.")


### Cell 7.9 — optional old-clustering geometry / angular comparison against its own singles pool

In [ ]:
# Cell 7.9 — optional old-clustering geometry / angular comparison against its own singles pool
plot_all_delta_occupancies(
    obs_doublets_old["dx"], obs_doublets_old["dy"], obs_doublets_old["dz"],
    prefix="Observed old DBSCAN doublets (from summary)", bins=70, symmetric=False,
)
plot_all_delta_occupancies(
    singles_mc_old["dx"], singles_mc_old["dy"], singles_mc_old["dz"],
    prefix="Singles-driven MC from old DBSCAN singles", bins=70, symmetric=False,
)

overlay_hist_simple(
    obs_doublets_old["theta_zx"], singles_mc_old["theta_zx"],
    "Observed old DBSCAN doublets", "Singles-driven MC",
    title="theta_zx — observed old DBSCAN doublets vs singles-driven MC",
    xlabel=r"$\theta_{zx}$ [rad]", bins=60,
)

overlay_hist_simple(
    obs_doublets_old["theta_zy"], singles_mc_old["theta_zy"],
    "Observed old DBSCAN doublets", "Singles-driven MC",
    title="theta_zy — observed old DBSCAN doublets vs singles-driven MC",
    xlabel=r"$\theta_{zy}$ [rad]", bins=60,
)

overlay_hist_simple(
    obs_doublets_old["theta_z"], singles_mc_old["theta_z"],
    "Observed old DBSCAN doublets", "Singles-driven MC",
    title="theta_z — observed old DBSCAN doublets vs singles-driven MC",
    xlabel=r"$\theta_{z}$ [rad]", bins=60,
)


# Cell 8 — overlays

In [ ]:
overlay_hist(theta_zx_mc, theta_zx_real, theta_zx_old, nbins=60, title="theta_zx (MC continuous vs data)", xlabel=r"$\theta_{zx}$ (rad)")
overlay_hist(theta_zy_mc, theta_zy_real, theta_zy_old, nbins=60, title="theta_zy (MC continuous vs data)", xlabel=r"$\theta_{zy}$ (rad)")
overlay_hist(theta_z_mc,  theta_z_real, theta_z_old, nbins=60, title="theta_z (MC continuous vs data)",  xlabel=r"$\theta_{z}$ (rad)")

overlay_hist(theta_zx_vox, theta_zx_real, theta_zx_old, nbins=60, title="theta_zx (MC voxelized vs data)", xlabel=r"$\theta_{zx}$ (rad)")
overlay_hist(theta_zy_vox, theta_zy_real, theta_zy_old, nbins=60, title="theta_zy (MC voxelized vs data)", xlabel=r"$\theta_{zy}$ (rad)")
overlay_hist(theta_z_vox,  theta_z_real, theta_z_old,  nbins=60, title="theta_z (MC voxelized vs data)",  xlabel=r"$\theta_{z}$ (rad)")

# Cell 9 — delta-space helpers

These helpers let you inspect the doublet displacement space directly:
- data and MC in **Δx vs Δz**
- data and MC in **Δz vs Δy**
- projected separations in **xz** and **zy**

This is useful to diagnose:
- missing small-separation pairs
- preferred lattice directions
- clustering / centroiding effects
- anisotropies that may not be obvious in angle space


In [ ]:

def load_data_doublet_deltas(
    h5_path,
    min_dist=10.0,
    use_centroid=False,
    sort_by_z=False,
):
    """
    Return per-doublet displacement vectors for accepted data events.

    Outputs:
        dx, dy, dz
    where
        dx = x2 - x1
        dy = y2 - y1
        dz = z2 - z1

    If sort_by_z=True, points are ordered so z2 >= z1.
    """
    dxs, dys, dzs = [], [], []

    with h5py.File(h5_path, "r") as f:
        for key in f["events"]:
            g = f["events"][key]
            labels = g["labels"][:]
            x = g["x"][:]
            y = g["y"][:]
            z = g["z"][:]

            geom_mask = (
                (y >= -51.85) & (y <= 51.85) &
                (((z >= 12.68) & (z <= 54.32)) | ((z >= -54.32) & (z <= -12.68)))
            )

            idx = np.where(geom_mask & (labels >= 0))[0]
            if idx.size < 2:
                continue

            labs = labels[idx]
            uniq = np.unique(labs)
            if uniq.size != 2:
                continue

            c0 = idx[labs == uniq[0]]
            c1 = idx[labs == uniq[1]]

            if use_centroid:
                p0 = np.array([x[c0].mean(), y[c0].mean(), z[c0].mean()], dtype=float)
                p1 = np.array([x[c1].mean(), y[c1].mean(), z[c1].mean()], dtype=float)
            else:
                p0 = np.array([x[c0[0]], y[c0[0]], z[c0[0]]], dtype=float)
                p1 = np.array([x[c1[0]], y[c1[0]], z[c1[0]]], dtype=float)

            d = p1 - p0
            dist = np.linalg.norm(d)
            if dist < float(min_dist):
                continue

            if sort_by_z and (p1[2] < p0[2]):
                p0, p1 = p1, p0

            d = p1 - p0
            dxs.append(d[0])
            dys.append(d[1])
            dzs.append(d[2])

    return np.asarray(dxs), np.asarray(dys), np.asarray(dzs)


def deltas_from_stored_tracks(stored_tracks, sort_by_z=False):
    dx, dy, dz = [], [], []

    for t in stored_tracks:
        p0 = np.array([t[0], t[1], t[2]], dtype=float)
        p1 = np.array([t[3], t[4], t[5]], dtype=float)

        if sort_by_z and (p1[2] < p0[2]):
            p0, p1 = p1, p0

        d = p1 - p0
        dx.append(d[0])
        dy.append(d[1])
        dz.append(d[2])

    return np.asarray(dx), np.asarray(dy), np.asarray(dz)


def projected_separations(dx, dy, dz):
    rho_xz = np.sqrt(dx*dx + dz*dz)
    rho_zy = np.sqrt(dz*dz + dy*dy)
    return rho_xz, rho_zy


# Cell 10 — delta-space plotting helpers

In [ ]:

def plot_delta_occupancy_2d(a, b, xlabel, ylabel, title, bins=80, symmetric=True, equal_aspect=True):
    plt.figure(figsize=(7, 6))

    if len(a) == 0 or len(b) == 0:
        print(f"No entries for: {title}")
        return

    if symmetric:
        lim = max(np.max(np.abs(a)), np.max(np.abs(b)))
        ranges = [[-lim, lim], [-lim, lim]]
    else:
        ranges = None

    plt.hist2d(a, b, bins=bins, range=ranges, cmap="viridis")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.colorbar(label="counts")
    if equal_aspect:
        plt.gca().set_aspect("equal", adjustable="box")
    plt.tight_layout()
    save_current_figure(title)
    plt.show()


def plot_all_delta_occupancies(dx, dy, dz, prefix="Sample", bins=80, symmetric=True):
    plot_delta_occupancy_2d(
        dx, dz,
        xlabel=r"$\Delta x$ [cm]",
        ylabel=r"$\Delta z$ [cm]",
        title=f"{prefix}: $\Delta x$ vs $\Delta z$",
        bins=bins,
        symmetric=symmetric,
    )

    plot_delta_occupancy_2d(
        dz, dy,
        xlabel=r"$\Delta z$ [cm]",
        ylabel=r"$\Delta y$ [cm]",
        title=f"{prefix}: $\Delta z$ vs $\Delta y$",
        bins=bins,
        symmetric=symmetric,
    )


def plot_projected_separations(rho_xz, rho_zy, title="Projected doublet separation", bins=50, density=False):
    plt.figure(figsize=(7, 5))
    plt.hist(rho_xz, bins=bins, density=density, histtype="step", linewidth=2,
             label=r"$\sqrt{\Delta x^2 + \Delta z^2}$")
    plt.hist(rho_zy, bins=bins, density=density, histtype="step", linewidth=2,
             label=r"$\sqrt{\Delta z^2 + \Delta y^2}$")
    plt.xlabel("projected separation [cm]")
    plt.ylabel("normalized counts" if density else "counts")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    save_current_figure(title)
    plt.show()


def overlay_projected_separations(rho_xz_data, rho_zy_data, rho_xz_mc, rho_zy_mc, title="", bins=50):
    fig, ax = plt.subplots(figsize=(8, 5))

    allv = np.concatenate([rho_xz_data, rho_zy_data, rho_xz_mc, rho_zy_mc])
    edges = np.linspace(float(np.min(allv)), float(np.max(allv)), bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths = np.diff(edges)

    def _density_with_err(data):
        counts, _ = np.histogram(data, bins=edges)
        dens = counts / (max(len(data), 1) * widths)
        err = np.sqrt(counts) / (max(len(data), 1) * widths)
        return counts, dens, err

    c_xz_d, d_xz_d, e_xz_d = _density_with_err(rho_xz_data)
    c_zy_d, d_zy_d, e_zy_d = _density_with_err(rho_zy_data)
    c_xz_m, _, _ = _density_with_err(rho_xz_mc)
    c_zy_m, _, _ = _density_with_err(rho_zy_mc)

    scale_xz = c_xz_d.sum() / max(c_xz_m.sum(), 1)
    scale_zy = c_zy_d.sum() / max(c_zy_m.sum(), 1)

    chi2_xz, ndf_xz, p_xz, _ = chi2_ndf_pvalue_from_hists(c_xz_d, c_xz_m * scale_xz, n_fit_params=1)
    chi2_zy, ndf_zy, p_zy, _ = chi2_ndf_pvalue_from_hists(c_zy_d, c_zy_m * scale_zy, n_fit_params=1)

    ax.errorbar(centers, d_xz_d, yerr=e_xz_d, fmt="o", markersize=4, capsize=2,
                label=fr"Data xz | {_stats_label(chi2_xz, ndf_xz, p_xz)}")
    ax.errorbar(centers, d_zy_d, yerr=e_zy_d, fmt="s", markersize=4, capsize=2,
                label=fr"Data zy | {_stats_label(chi2_zy, ndf_zy, p_zy)}")

    ax.hist(rho_xz_mc, bins=edges, density=True, histtype="step", linewidth=2, linestyle="--",
            label=r"MC xz")
    ax.hist(rho_zy_mc, bins=edges, density=True, histtype="step", linewidth=2, linestyle="--",
            label=r"MC zy")

    ax.set_xlabel("projected separation [cm]")
    ax.set_ylabel("normalized counts")
    ax.set_title(f"Projected doublet separation: data vs MC - {title}")
    ax.legend(fontsize=8)
    plt.tight_layout()
    save_current_figure(f"Projected_doublet_separation_data_vs_MC_{title}")
    plt.show()

    print("Projected doublet separation: data vs MC")
    print(f"  xz: {_stats_label(chi2_xz, ndf_xz, p_xz)}")
    print(f"  zy: {_stats_label(chi2_zy, ndf_zy, p_zy)}")


def two_panel_projected_comparison(rho_xz_data, rho_zy_data, rho_xz_mc, rho_zy_mc, titleGeneral="", bins=50):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    datasets = [
        (axes[0], rho_xz_data, rho_xz_mc, "xz projection", "projected separation in xz [cm]"),
        (axes[1], rho_zy_data, rho_zy_mc, "zy projection", "projected separation in zy [cm]"),
    ]

    for ax, data, mc, title, xlabel in datasets:
        allv = np.concatenate([data, mc])
        edges = np.linspace(float(np.min(allv)), float(np.max(allv)), bins + 1)
        centers = 0.5 * (edges[:-1] + edges[1:])
        widths = np.diff(edges)

        counts_d, _ = np.histogram(data, bins=edges)
        counts_m, _ = np.histogram(mc, bins=edges)

        dens_d = counts_d / (max(len(data), 1) * widths)
        err_d = np.sqrt(counts_d) / (max(len(data), 1) * widths)

        scale_m = counts_d.sum() / max(counts_m.sum(), 1)
        chi2, ndf, pval, _ = chi2_ndf_pvalue_from_hists(counts_d, counts_m * scale_m, n_fit_params=1)

        ax.errorbar(centers, dens_d, yerr=err_d, fmt="o", markersize=4, capsize=2,
                    label=f"Data | {_stats_label(chi2, ndf, pval)}")
        ax.hist(mc, bins=edges, density=True, histtype="step", linewidth=2, linestyle="--", label="MC")
        ax.set_xlabel(xlabel)
        ax.set_title(title)
        ax.legend(fontsize=8)

    axes[0].set_ylabel("normalized counts")
    plt.tight_layout()
    plt.suptitle(titleGeneral)
    save_current_figure(f"Projected_separation_two_panel_comparison_{titleGeneral}")
    plt.show()


def ratio_projected_zy_over_xz(rho_xz_data, rho_zy_data, rho_xz_mc, rho_zy_mc, title="", nbins=40):
    maxv = max(np.max(rho_xz_data), np.max(rho_zy_data), np.max(rho_xz_mc), np.max(rho_zy_mc))
    bins = np.linspace(0, maxv, nbins)

    h_xz_data, _ = np.histogram(rho_xz_data, bins=bins, density=True)
    h_zy_data, _ = np.histogram(rho_zy_data, bins=bins, density=True)
    h_xz_mc,   _ = np.histogram(rho_xz_mc,   bins=bins, density=True)
    h_zy_mc,   _ = np.histogram(rho_zy_mc,   bins=bins, density=True)

    centers = 0.5 * (bins[:-1] + bins[1:])
    eps = 1e-12

    ratio_data = h_zy_data / (h_xz_data + eps)
    ratio_mc   = h_zy_mc   / (h_xz_mc   + eps)

    plt.figure(figsize=(7, 5))
    plt.plot(centers, ratio_data, marker="o", label="Data: zy / xz")
    plt.plot(centers, ratio_mc, marker="o", label="MC: zy / xz")
    plt.axhline(1.0, color="k", linestyle="--", alpha=0.5)
    plt.xlabel("projected separation [cm]")
    plt.ylabel("ratio")
    plt.title(f"Relative suppression/enhancement: zy vs xz - {title}")
    plt.legend()
    plt.tight_layout()
    save_current_figure(f"Projected_separation_ratio_zy_over_xz_{title}")
    plt.show()


# Cell 11 — build delta-space samples from data and MC

This cell computes:
- data deltas
- MC deltas from the stored continuous-MC sample
- an optional **same-module-only** MC subset
- projected separations for both


In [ ]:
def _extract_two_points_from_track(t):
    """
    Extract p0, p1 from several possible stored_tracks formats.

    Supported formats:
      - (x1, y1, z1, x2, y2, z2)
      - [x1, y1, z1, x2, y2, z2]
      - {"x1":..., "y1":..., "z1":..., "x2":..., "y2":..., "z2":...}
      - {"p0":[x1,y1,z1], "p1":[x2,y2,z2]}
      - {"a":[x1,y1,z1], "b":[x2,y2,z2]}
      - {"a":{"x":...,"y":...,"z":...}, "b":{"x":...,"y":...,"z":...}, ...}
    """
    # tuple/list/array: [x1,y1,z1,x2,y2,z2]
    if isinstance(t, (list, tuple, np.ndarray)):
        if len(t) >= 6:
            p0 = np.array([t[0], t[1], t[2]], dtype=float)
            p1 = np.array([t[3], t[4], t[5]], dtype=float)
            return p0, p1
        raise ValueError(f"Track sequence has length {len(t)} < 6")

    if isinstance(t, dict):
        keys = set(t.keys())

        # flat dict
        if {"x1", "y1", "z1", "x2", "y2", "z2"}.issubset(keys):
            p0 = np.array([t["x1"], t["y1"], t["z1"]], dtype=float)
            p1 = np.array([t["x2"], t["y2"], t["z2"]], dtype=float)
            return p0, p1

        # p0/p1 vectors
        if {"p0", "p1"}.issubset(keys):
            p0 = np.array(t["p0"], dtype=float)
            p1 = np.array(t["p1"], dtype=float)
            return p0, p1

        # a/b style
        if {"a", "b"}.issubset(keys):
            a = t["a"]
            b = t["b"]

            # nested dicts with x,y,z
            if isinstance(a, dict) and isinstance(b, dict):
                if {"x", "y", "z"}.issubset(a.keys()) and {"x", "y", "z"}.issubset(b.keys()):
                    p0 = np.array([a["x"], a["y"], a["z"]], dtype=float)
                    p1 = np.array([b["x"], b["y"], b["z"]], dtype=float)
                    return p0, p1

            # already simple vectors
            p0 = np.array(a, dtype=float)
            p1 = np.array(b, dtype=float)
            return p0, p1

        raise ValueError(f"Unrecognized dict track format with keys: {sorted(keys)}")

    raise TypeError(f"Unsupported track type: {type(t)}")


def _track_modules_if_available(t):
    """
    Return (module_a, module_b) when present in stored_tracks, else (None, None).
    """
    if isinstance(t, dict) and "a" in t and "b" in t:
        a = t["a"]
        b = t["b"]
        if isinstance(a, dict) and isinstance(b, dict):
            return a.get("module", None), b.get("module", None)
    return None, None


def deltas_from_stored_tracks(stored_tracks, sort_by_z=False, verbose=True, same_module_only=False):
    dx, dy, dz = [], [], []

    if verbose:
        try:
            print("Example stored_tracks entry:", stored_tracks[0])
        except Exception:
            pass

    for i, t in enumerate(stored_tracks):
        try:
            p0, p1 = _extract_two_points_from_track(t)
        except Exception as e:
            print(f"Skipping stored_tracks[{i}] because it could not be parsed: {e}")
            continue

        if same_module_only:
            ma, mb = _track_modules_if_available(t)
            if (ma is not None) and (mb is not None) and (ma != mb):
                continue

        if sort_by_z and (p1[2] < p0[2]):
            p0, p1 = p1, p0

        d = p1 - p0
        dx.append(d[0])
        dy.append(d[1])
        dz.append(d[2])

    return np.asarray(dx), np.asarray(dy), np.asarray(dz)


# --- build data deltas ---
dx_data, dy_data, dz_data = load_data_doublet_deltas(
    "normal_clusters.h5",
    min_dist=min_dist,
    use_centroid=True,
    sort_by_z=False,
)

dx_data_ord, dy_data_ord, dz_data_ord = load_data_doublet_deltas(
    "normal_clusters.h5",
    min_dist=min_dist,
    use_centroid=True,
    sort_by_z=True,
)

print("Accepted data doublets:", len(dx_data))


# --- build old-clustering data deltas ---
dx_old, dy_old, dz_old = load_data_doublet_deltas(
    "HC/test_hot.h5",
    min_dist=min_dist,
    use_centroid=True,
    sort_by_z=False,
)

dx_old_ord, dy_old_ord, dz_old_ord = load_data_doublet_deltas(
    "HC/test_hot.h5",
    min_dist=min_dist,
    use_centroid=True,
    sort_by_z=True,
)

print("Accepted old-clustering doublets:", len(dx_old))

# --- build MC deltas ---
dx_mc, dy_mc, dz_mc = deltas_from_stored_tracks(
    stored_tracks, sort_by_z=False, verbose=True, same_module_only=False
)
dx_mc_ord, dy_mc_ord, dz_mc_ord = deltas_from_stored_tracks(
    stored_tracks, sort_by_z=True, verbose=False, same_module_only=False
)

# optional same-module-only MC view
dx_mc_same, dy_mc_same, dz_mc_same = deltas_from_stored_tracks(
    stored_tracks, sort_by_z=False, verbose=False, same_module_only=True
)
dx_mc_same_ord, dy_mc_same_ord, dz_mc_same_ord = deltas_from_stored_tracks(
    stored_tracks, sort_by_z=True, verbose=False, same_module_only=True
)

print("Accepted MC stored doublets (all):", len(dx_mc))
print("Accepted MC stored doublets (same module only):", len(dx_mc_same))

# --- projected separations ---
rho_xz_data = np.sqrt(dx_data**2 + dz_data**2)
rho_zy_data = np.sqrt(dz_data**2 + dy_data**2)

rho_xz_old = np.sqrt(dx_old**2 + dz_old**2)
rho_zy_old = np.sqrt(dz_old**2 + dy_old**2)

rho_xz_mc = np.sqrt(dx_mc**2 + dz_mc**2)
rho_zy_mc = np.sqrt(dz_mc**2 + dy_mc**2)

rho_xz_mc_same = np.sqrt(dx_mc_same**2 + dz_mc_same**2)
rho_zy_mc_same = np.sqrt(dz_mc_same**2 + dy_mc_same**2)

print("Done building delta-space variables.")

# Cell 12 — delta-space occupancy plots

In [ ]:

plot_all_delta_occupancies(
    dx_data, dy_data, dz_data,
    prefix="Data_new (signed deltas)",
    bins=70,
    symmetric=True,
)

plot_all_delta_occupancies(
    dx_old, dy_old, dz_old,
    prefix="Data_old_DBSCAN (signed deltas)",
    bins=70,
    symmetric=True,
)

plot_all_delta_occupancies(
    dx_mc, dy_mc, dz_mc,
    prefix="MC continuous (signed deltas)",
    bins=70,
    symmetric=True,
)


# Cell 13 — ordered-by-z delta-space occupancy plots

Ordering each doublet so that $z_2 \ge z_1$ makes $\Delta z \ge 0$ and often gives a cleaner diagnostic view.


In [ ]:

plot_all_delta_occupancies(
    dx_data_ord, dy_data_ord, dz_data_ord,
    prefix="Data_new (ordered by z)",
    bins=70,
    symmetric=False,
)

plot_all_delta_occupancies(
    dx_old_ord, dy_old_ord, dz_old_ord,
    prefix="Data_old_DBSCAN (ordered by z)",
    bins=70,
    symmetric=False,
)

plot_all_delta_occupancies(
    dx_mc_ord, dy_mc_ord, dz_mc_ord,
    prefix="MC continuous (ordered by z)",
    bins=70,
    symmetric=False,
)


# Cell 14 — projected separation plots

The last two blocks compare data against:
- all stored MC doublets
- same-module-only stored MC doublets


In [ ]:

plot_projected_separations(
    rho_xz_data, rho_zy_data,
    title="Data_new projected doublet separation",
    bins=50,
    density=False,
)

plot_projected_separations(
    rho_xz_old, rho_zy_old,
    title="Data_old_DBSCAN projected doublet separation",
    bins=50,
    density=False,
)

plot_projected_separations(
    rho_xz_mc, rho_zy_mc,
    title="MC projected doublet separation (all stored pairs)",
    bins=50,
    density=False,
)

plot_projected_separations(
    rho_xz_mc_same, rho_zy_mc_same,
    title="MC projected doublet separation (same-module only)",
    bins=50,
    density=False,
)

overlay_projected_separations(
    rho_xz_data, rho_zy_data,
    rho_xz_mc, rho_zy_mc,
    "new_clustering",
    bins=50,
)

two_panel_projected_comparison(
    rho_xz_data, rho_zy_data,
    rho_xz_mc, rho_zy_mc,
    "new_clustering",
    bins=50,
)

ratio_projected_zy_over_xz(
    rho_xz_data, rho_zy_data,
    rho_xz_mc, rho_zy_mc,
    "new_clustering",
    nbins=40,
)

overlay_projected_separations(
    rho_xz_old, rho_zy_old,
    rho_xz_mc, rho_zy_mc,
    "old_clustering",
    bins=50,
)

two_panel_projected_comparison(
    rho_xz_old, rho_zy_old,
    rho_xz_mc, rho_zy_mc,
    "old_clustering",
    bins=50,
)

ratio_projected_zy_over_xz(
    rho_xz_old, rho_zy_old,
    rho_xz_mc, rho_zy_mc,
    "old_clustering",
    nbins=40,
)

overlay_projected_separations(
    rho_xz_data, rho_zy_data,
    rho_xz_mc_same, rho_zy_mc_same,
    "new_clustering",
    bins=50,
)

two_panel_projected_comparison(
    rho_xz_data, rho_zy_data,
    rho_xz_mc_same, rho_zy_mc_same,
    "new_clustering",
    bins=50,
)

ratio_projected_zy_over_xz(
    rho_xz_data, rho_zy_data,
    rho_xz_mc_same, rho_zy_mc_same,
    "new_clustering",
    nbins=40,
)

overlay_projected_separations(
    rho_xz_old, rho_zy_old,
    rho_xz_mc_same, rho_zy_mc_same,
    "old_clustering",
    bins=50,
)

two_panel_projected_comparison(
    rho_xz_old, rho_zy_old,
    rho_xz_mc_same, rho_zy_mc_same,
    "old_clustering",
    bins=50,
)

ratio_projected_zy_over_xz(
    rho_xz_old, rho_zy_old,
    rho_xz_mc_same, rho_zy_mc_same,
    "old_clustering",
    nbins=40,
)
